In [0]:
dbutils.widgets.text("run_id", "")
job_run_id = dbutils.widgets.get("run_id")

if not job_run_id:
    raise ValueError("job_run_id is required")

print("Job Run ID:", job_run_id)

Job Run ID: 12345


In [0]:
audit_df = spark.sql(f"""
SELECT
    p.table_id AS Table_ID,
    t.table_name AS Table_Name,
    p.layer AS Layer,
    p.start_time AS Start_Time,
    p.end_time AS End_Time,
    p.status AS Status,
    p.number_of_records AS Number_Of_Records,
    p.error_message AS Error_Message
FROM banking.metadata.pipeline_runs p
LEFT JOIN banking.metadata.tables t
    ON p.table_id = t.table_id
WHERE p.run_id = {job_run_id}
ORDER BY p.table_id
""")

display(audit_df)

Table_ID,Table_Name,Layer,Start_Time,End_Time,Status,Number_Of_Records,Error_Message
1,customers,Gold,2026-11-16T13:38:29.754Z,null,INPROGRESS,null,null
5,credit_bureau_reports,Silver,2026-11-16T12:15:29.239Z,2026-11-16T12:16:19.571Z,SUCCESS,4000,null
6,payment_gateway_logs,Silver,2026-11-16T12:16:52.820Z,2026-11-16T12:17:26.532Z,SUCCESS,15000,null
7,customer_360,Gold,2026-11-16T14:01:47.370Z,2026-11-16T14:02:27.713Z,SUCCESS,8000,null
8,branch_performance,Gold,2026-11-16T14:03:18.923Z,2026-11-16T14:03:44.641Z,SUCCESS,4,null
9,transaction_channel_summary,Gold,2026-11-16T14:03:54.050Z,2026-11-16T14:04:19.769Z,SUCCESS,5716,null
10,daily_bank_kpi,Gold,2026-11-16T14:04:28.728Z,2026-11-16T14:04:54.924Z,SUCCESS,396,null
11,risk_customer_summary,Gold,2026-11-16T14:02:41.225Z,2026-11-16T14:03:07.151Z,SUCCESS,3,null


In [0]:
from pyspark.sql.functions import col

total = audit_df.count()
failed = audit_df.filter(col("status") != "SUCCESS").count()

if failed > 0:
    overall_status = "FAILED"
else:
    overall_status = "SUCCESS"

print("Total Tables:", total)
print("Failed Tables:", failed)
print("Overall Status:", overall_status)

Total Tables: 8
Failed Tables: 1
Overall Status: FAILED


In [0]:
pdf = audit_df.toPandas()

html_table = pdf.to_html(
    index=False,
    border=1,
    justify="center",
    classes="table table-striped"
)

print(html_table)

<table border="1" class="dataframe table table-striped">
  <thead>
    <tr style="text-align: center;">
      <th>Table_ID</th>
      <th>Table_Name</th>
      <th>Layer</th>
      <th>Start_Time</th>
      <th>End_Time</th>
      <th>Status</th>
      <th>Number_Of_Records</th>
      <th>Error_Message</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>1</td>
      <td>customers</td>
      <td>Gold</td>
      <td>2026-11-16 13:38:29.754570</td>
      <td>NaT</td>
      <td>INPROGRESS</td>
      <td>NaN</td>
      <td>None</td>
    </tr>
    <tr>
      <td>5</td>
      <td>credit_bureau_reports</td>
      <td>Silver</td>
      <td>2026-11-16 12:15:29.239488</td>
      <td>2026-11-16 12:16:19.571423</td>
      <td>SUCCESS</td>
      <td>4000.0</td>
      <td>None</td>
    </tr>
    <tr>
      <td>6</td>
      <td>payment_gateway_logs</td>
      <td>Silver</td>
      <td>2026-11-16 12:16:52.820819</td>
      <td>2026-11-16 12:17:26.532613</td>
      <td>SUCCESS</td>
      <td>15000.0<

In [0]:
body = f"""
<html>
<body>

<p><b>This is an auto-generated email. Please do not reply.</b></p>

<p>
End-to-end run for <b>Source → Bronze → Silver → Gold</b> layers has completed.
Below is the execution summary:
</p>

<br>

{html_table}

<br>

<p>
For any queries please reach out to:
<b>ranayuvrajsingh1111@gmail.com</b>
</p>

<br>

<p>
Thanks<br>
<b>X Bank Support Team</b>
</p>

</body>
</html>
"""

print(body)



<html>
<body>

<p><b>This is an auto-generated email. Please do not reply.</b></p>

<p>
End-to-end run for <b>Source → Bronze → Silver → Gold</b> layers has completed.
Below is the execution summary:
</p>

<br>

<table border="1" class="dataframe table table-striped">
  <thead>
    <tr style="text-align: center;">
      <th>Table_ID</th>
      <th>Table_Name</th>
      <th>Layer</th>
      <th>Start_Time</th>
      <th>End_Time</th>
      <th>Status</th>
      <th>Number_Of_Records</th>
      <th>Error_Message</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>1</td>
      <td>customers</td>
      <td>Gold</td>
      <td>2026-11-16 13:38:29.754570</td>
      <td>NaT</td>
      <td>INPROGRESS</td>
      <td>NaN</td>
      <td>None</td>
    </tr>
    <tr>
      <td>5</td>
      <td>credit_bureau_reports</td>
      <td>Silver</td>
      <td>2026-11-16 12:15:29.239488</td>
      <td>2026-11-16 12:16:19.571423</td>
      <td>SUCCESS</td>
      <td>4000.0</td>
      <td>None</td>
    </

In [0]:
subject = f"X Bank | {overall_status} | End to End Run"

print(subject)

X Bank | FAILED | End to End Run


In [0]:
gmail_api_key = dbutils.secrets.get(
            scope="banking-scope",
            key="gmail_api_key"
        )

In [0]:
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

# Gmail account
EMAIL = "ranayuvrajsingh1111@gmail.com"
APP_PASSWORD = gmail_api_key

# Receiver
to_email = "ranayuvrajsingh1111@gmail.com"

# Create message
msg = MIMEMultipart()
msg["From"] = EMAIL
msg["To"] = to_email
msg["Subject"] = subject

msg.attach(MIMEText(body, "html"))

try:
    with smtplib.SMTP("smtp.gmail.com", 587) as server:
        server.starttls()
        server.login(EMAIL, APP_PASSWORD)
        server.send_message(msg)

    print("✅ Email sent successfully!")

except Exception as e:
    print("❌ Error sending email:", e)